# 论文表格整理（来自 `实验结果记录 - 论文表格.csv`）

此 notebook 会：
- 自动切分 CSV 中的多个表（表1~表5）
- 生成可直接用于论文的 Markdown / LaTeX 表格
- 从多维度汇总与对比，标注我们方法（默认以 `DistRL` 作为前缀）优势

如需修改“我们方法”集合或指标范围，改下面的配置即可。


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)

DATA_PATH = Path("../data/实验结果记录 - 论文表格.csv")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

OUR_PREFIXES = ["DistRL"]  # 你可以按需补充，例如 ["DistRL", "IQN(SR)"]
BASELINES = ["PPO", "GRPO", "base model"]  # 基线集合

raw = pd.read_csv(DATA_PATH, header=None)
raw.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

FIG_DIR = OUTPUT_DIR / "figs"
FIG_DIR.mkdir(exist_ok=True)
SAVE_FIGS = True  # set False to skip writing image files


def savefig(name: str):
    if SAVE_FIGS:
        out = FIG_DIR / f"{name}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=200)
        print("saved:", out)
    plt.show()


In [ ]:
def _is_table_title(x):
    return isinstance(x, str) and x.strip().startswith("表")

# 找到各表起始行（对首列做 strip 处理）
raw0 = raw.copy()
raw0[0] = raw0[0].astype(str).str.strip()
section_starts = [i for i, v in enumerate(raw0[0]) if _is_table_title(v)]
section_starts


In [ ]:
def _clean_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # normalize strings: strip spaces and unify common whitespace
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    out = out.replace("#DIV/0!", np.nan)
    return out

clean = _clean_df(raw)


In [ ]:
def _build_columns_from_table1(header_block: pd.DataFrame) -> list:
    # header_block: 3 rows, columns 2..23
    row1 = header_block.iloc[0].copy()
    row2 = header_block.iloc[1].copy()
    row3 = header_block.iloc[2].copy()

    # dataset 名称向右填充
    row1 = row1.ffill()

    cols = []
    for i in range(len(row1)):
        dataset = row1.iloc[i]
        metric2 = row2.iloc[i]
        metric3 = row3.iloc[i]

        metric = metric3 if pd.notna(metric3) else metric2
        if pd.isna(metric):
            metric = ""
        if pd.isna(dataset):
            dataset = ""

        if metric:
            col = f"{dataset} {metric}".strip()
        else:
            col = f"{dataset}".strip()
        cols.append(col)
    return cols

# 表1的列结构（后续表2~5沿用）
start = section_starts[0]
header_block = clean.iloc[start+1:start+4, 2:24]
metric_cols = _build_columns_from_table1(header_block)
metric_cols


In [ ]:
def _extract_block(start_row: int, end_row: int):
    block = clean.iloc[start_row:end_row].copy()
    return block

# 各表分块
sections = {}
for idx, start in enumerate(section_starts):
    end = section_starts[idx+1] if idx+1 < len(section_starts) else len(clean)
    title = clean.iloc[start, 0]
    sections[title] = _extract_block(start, end)

list(sections.keys())


In [ ]:
def _build_table_from_block(block: pd.DataFrame, data_start_row: int, data_end_row: int, use_group=False):
    # data rows assume columns: 0(group), 1(method), 2..23 metrics
    data = block.iloc[data_start_row:data_end_row].copy()
    data = data.dropna(how="all")

    # 设置列名
    cols = ["group", "method"] + metric_cols
    data = data.iloc[:, :2+len(metric_cols)]
    data.columns = cols

    # 过滤空method行
    data = data[pd.notna(data["method"])]

    if not use_group:
        data = data.drop(columns=["group"])

    # 数值列转换
    for c in metric_cols:
        data[c] = pd.to_numeric(data[c], errors="coerce")
    return data


In [ ]:
# 表1：均值与标准差
block1 = sections[clean.iloc[section_starts[0], 0]]
# 定位 "均值记录" 与 "std记录"
rows = block1.reset_index(drop=True)
col0 = rows[0].astype(str).str.strip()

idx_mean_list = col0[col0 == "均值记录"].index.tolist()
idx_std_list = col0[col0.str.lower().str.startswith("std记录")].index.tolist()

if not idx_mean_list:
    raise ValueError("未找到 '均值记录' 行，请检查表1格式或字符串空格。")
idx_mean = idx_mean_list[0]

# 如果未找到 std 记录，则用 block 末尾作为 std 截止，并提示
if not idx_std_list:
    idx_std = len(rows)
    print("Warning: 未找到 'std记录' 行，使用表1末尾作为 std 起点/终点。")
else:
    idx_std = idx_std_list[0]

# 均值行：从 idx_mean 到 idx_std 之前的非空method
mean_tbl = _build_table_from_block(rows, idx_mean, idx_std, use_group=True)

# 标准差行：从 idx_std 到 block 末尾
std_tbl = _build_table_from_block(rows, idx_std, len(rows), use_group=True)

mean_tbl.head(), std_tbl.head()


In [ ]:
def _norm_method(m):
    return m.strip() if isinstance(m, str) else m

def _is_ours(method: str) -> bool:
    m = _norm_method(method)
    return isinstance(m, str) and any(m.startswith(p) for p in OUR_PREFIXES)

# 主表：我们方法 vs 基线
mean_tbl["method_norm"] = mean_tbl["method"].apply(_norm_method)
mean_tbl["is_ours"] = mean_tbl["method_norm"].apply(_is_ours)
mean_tbl["is_baseline"] = mean_tbl["method_norm"].str.lower().isin([b.lower() for b in BASELINES])
mean_tbl


In [ ]:
# 便于论文的核心表：只保留均值 & 重要列
key_cols = [c for c in metric_cols if c and ("avg" in c or "p@" in c or c in ["AVG", "AVGP"])]
main_table = mean_tbl[["group", "method"] + key_cols].copy()
main_table.head()


In [ ]:
# Plot: key metrics bar charts (top methods)
plot_cols = []
plot_cols += [c for c in key_cols if ("AVG" in c or "avg" in c)][:4]
plot_cols += [c for c in key_cols if "p@" in c][:4]

for metric in plot_cols:
    tmp = main_table[["method", metric]].dropna().sort_values(metric, ascending=False).head(12)
    if tmp.empty:
        continue
    plt.figure(figsize=(12, 6))
    sns.barplot(data=tmp, x="method", y=metric)
    plt.title(f"Top methods by {metric}")
    plt.xlabel("method")
    plt.ylabel(metric)
    plt.xticks(rotation=45, ha="right")
    savefig(f"table1_top_{metric}")

# Heatmap (top methods by avg_mean) for paper-style overview
heat_cols = [c for c in key_cols if c]
heat_df = mean_tbl[["method"] + heat_cols].copy()
heat_df[heat_cols] = heat_df[heat_cols].apply(pd.to_numeric, errors="coerce")
# build avg_mean to select top methods
avg_cols = [c for c in heat_cols if "avg" in c.lower() or c in ["AVG", "AVGP"]]
if avg_cols:
    heat_df["avg_mean"] = heat_df[avg_cols].mean(axis=1, skipna=True)
    heat_df = heat_df.sort_values("avg_mean", ascending=False).head(12)
    heat_df = heat_df.set_index("method")[heat_cols]
    if not heat_df.empty:
        plt.figure(figsize=(12, 6))
        sns.heatmap(heat_df, cmap="viridis", annot=False)
        plt.title("Top methods heatmap (key metrics)")
        plt.xlabel("metric")
        plt.ylabel("method")
        savefig("table1_heatmap_top_methods")


In [ ]:
def _best_per_column(df: pd.DataFrame, cols: list):
    best = {}
    for c in cols:
        if df[c].notna().any():
            best[c] = df[c].max()
    return best

best_all = _best_per_column(mean_tbl, key_cols)

# 标注“我们方法”是否达到最优
advantage_flags = {}
for c, best_val in best_all.items():
    ours_best = mean_tbl.loc[mean_tbl["is_ours"], c].max()
    advantage_flags[c] = (pd.notna(ours_best) and np.isclose(ours_best, best_val))

advantage_flags


In [ ]:
# 生成一张“我们方法优势”表：和最强基线的差值（正值=优势）
adv_rows = []
for c in key_cols:
    ours_best = mean_tbl.loc[mean_tbl["is_ours"], c].max()
    base_best = mean_tbl.loc[mean_tbl["is_baseline"], c].max()
    if pd.notna(ours_best) and pd.notna(base_best):
        adv_rows.append({
            "metric": c,
            "ours_best": ours_best,
            "best_baseline": base_best,
            "delta": ours_best - base_best,
            "relative_%": (ours_best / base_best - 1) * 100 if base_best != 0 else np.nan,
        })

adv_table = pd.DataFrame(adv_rows).sort_values("delta", ascending=False)
adv_table


In [ ]:
# Plot: advantage over best baseline (delta)
if not adv_table.empty:
    tmp = adv_table.sort_values("delta", ascending=False).head(12)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=tmp, x="metric", y="delta")
    plt.title("Ours vs best baseline (delta)")
    plt.xlabel("metric")
    plt.ylabel("delta")
    plt.xticks(rotation=45, ha="right")
    savefig("table1_advantage_delta")


In [ ]:
# 输出 Markdown / LaTeX
main_md = main_table.to_markdown(index=False)
adv_md = adv_table.to_markdown(index=False)

print(main_md[:800])
print("
---
")
print(adv_md[:800])


In [ ]:
# 保存表格到 outputs/
(main_table).to_csv(OUTPUT_DIR / "table1_mean_main.csv", index=False)
(adv_table).to_csv(OUTPUT_DIR / "table1_ours_advantage.csv", index=False)

# LaTeX 版本
with open(OUTPUT_DIR / "table1_mean_main.tex", "w", encoding="utf-8") as f:
    f.write(main_table.to_latex(index=False))
with open(OUTPUT_DIR / "table1_ours_advantage.tex", "w", encoding="utf-8") as f:
    f.write(adv_table.to_latex(index=False))


In [ ]:
# 表2：三种 head 的实现方式（C51/QR/IQN）
block2 = sections[clean.iloc[section_starts[1],0]]
rows2 = block2.reset_index(drop=True)

# 数据从第 1 行开始直到空行
# 这里直接取 1..(遇到全空或下一个表) 作为数据块
# 手动截取 1..10 比较稳妥
head_tbl = _build_table_from_block(rows2, 1, 10, use_group=False)
head_tbl


In [ ]:
# 表2：汇总每类 head 在 AVG 上的表现
head_summary = head_tbl.copy()
head_summary["head"] = head_summary["method"].str.extract(r"^(C51|QR|IQN)")

cols_for_avg = [c for c in metric_cols if c in ["AVG", "AVGP"]]
head_summary = head_summary[["method", "head"] + cols_for_avg]
head_summary


In [ ]:
# Plot: head type comparison on AVG / AVGP
if 'head' in head_summary.columns:
    for col in [c for c in head_summary.columns if c in ["AVG", "AVGP"]]:
        tmp = head_summary[["head", col]].dropna()
        if tmp.empty:
            continue
        plt.figure(figsize=(8, 5))
        sns.barplot(data=tmp, x="head", y=col)
        plt.title(f"Head comparison on {col}")
        plt.xlabel("head")
        plt.ylabel(col)
        savefig(f"table2_head_{col}")


In [ ]:
# 表3：risk-sensitive
block3 = sections[clean.iloc[section_starts[2],0]]
rows3 = block3.reset_index(drop=True)

risk_tbl = _build_table_from_block(rows3, 1, 10, use_group=True)

# 只保留有数值的行
risk_tbl = risk_tbl.dropna(subset=key_cols, how='all')
risk_tbl


In [ ]:
# Plot: risk-sensitive comparisons
risk_cols = [c for c in key_cols if c in risk_tbl.columns]
if risk_cols:
    for col in risk_cols[:4]:
        tmp = risk_tbl[["group", "method", col]].dropna()
        if tmp.empty:
            continue
        plt.figure(figsize=(12, 6))
        sns.barplot(data=tmp, x="method", y=col, hue="group")
        plt.title(f"Risk-sensitive comparison: {col}")
        plt.xlabel("method")
        plt.ylabel(col)
        plt.xticks(rotation=45, ha="right")
        plt.legend(title="group", fontsize=9)
        savefig(f"table3_risk_{col}")


In [ ]:
# 表4：消融实验（当前数据较稀疏）
block4 = sections[clean.iloc[section_starts[3],0]]
rows4 = block4.reset_index(drop=True)

abla_tbl = _build_table_from_block(rows4, 1, len(rows4), use_group=False)
abla_tbl


In [ ]:
# Plot: ablation comparisons
abla_cols = [c for c in key_cols if c in abla_tbl.columns]
if abla_cols:
    for col in abla_cols[:4]:
        tmp = abla_tbl[["method", col]].dropna()
        if tmp.empty:
            continue
        plt.figure(figsize=(12, 6))
        sns.barplot(data=tmp, x="method", y=col)
        plt.title(f"Ablation comparison: {col}")
        plt.xlabel("method")
        plt.ylabel(col)
        plt.xticks(rotation=45, ha="right")
        savefig(f"table4_ablation_{col}")


In [ ]:
# 表5：不同 rollout 数量下对比
block5 = sections[clean.iloc[section_starts[4],0]]
rows5 = block5.reset_index(drop=True)

rollout_tbl = _build_table_from_block(rows5, 2, len(rows5), use_group=True)
rollout_tbl


In [ ]:
# Plot: rollout comparison
roll_cols = [c for c in key_cols if c in rollout_tbl.columns]
if roll_cols:
    for col in roll_cols[:4]:
        tmp = rollout_tbl[["group", "method", col]].dropna()
        if tmp.empty:
            continue
        plt.figure(figsize=(12, 6))
        sns.barplot(data=tmp, x="group", y=col, hue="method")
        plt.title(f"Rollout comparison: {col}")
        plt.xlabel("rollout")
        plt.ylabel(col)
        plt.xticks(rotation=0)
        plt.legend(title="method", fontsize=9)
        savefig(f"table5_rollout_{col}")


In [ ]:
# 生成紧凑对比表：按 avg 指标与 p@ 指标分别求均值
avg_cols = [c for c in metric_cols if "avg" in c and c not in ["AVGP"]]
pat_cols = [c for c in metric_cols if "p@" in c]

compact = mean_tbl[["method"] + avg_cols + pat_cols].copy()
compact["avg_mean"] = compact[avg_cols].mean(axis=1, skipna=True)
compact["p@_mean"] = compact[pat_cols].mean(axis=1, skipna=True)
compact = compact[["method", "avg_mean", "p@_mean"]].sort_values("avg_mean", ascending=False)
compact


In [ ]:
# 保存其他表
head_tbl.to_csv(OUTPUT_DIR / "table2_head.csv", index=False)
risk_tbl.to_csv(OUTPUT_DIR / "table3_risk_sensitive.csv", index=False)
abla_tbl.to_csv(OUTPUT_DIR / "table4_ablation.csv", index=False)
rollout_tbl.to_csv(OUTPUT_DIR / "table5_rollout.csv", index=False)
compact.to_csv(OUTPUT_DIR / "table_compact_avg_p.csv", index=False)
